# Patrón de Comportamiento: Strategy

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Strategy** define una **familia de algoritmos** intercambiables, los
encapsula y permite cambiarlos **en tiempo de ejecución** sin alterar el cliente.

### ¿Qué problema resuelve en la banca?
El **cálculo de intereses** depende del producto: una cuenta de **ahorros** rinde un
interés simple, un **CDT** aplica interés según plazo, y un **crédito de libranza** usa
otra fórmula. Meter todas las fórmulas en un `if/elif` dentro de la cuenta la vuelve
rígida y difícil de extender. Strategy encapsula cada fórmula por separado.

## Código *sin patrón* (el problema es evidente)
La clase `CuentaSinPatron` decide la fórmula con un `if/elif` según el tipo de producto.

In [6]:
class CuentaSinPatron:
    def __init__(self, saldo: float, producto: str):
        self.saldo = saldo
        self.producto = producto

    def calcular_interes(self, meses: int) -> float:
        if self.producto == "ahorros":
            return round(self.saldo * 0.02 * meses / 12, 2)
        elif self.producto == "cdt":
            return round(self.saldo * 0.08 * meses / 12, 2)
        elif self.producto == "libranza":
            return round(self.saldo * 0.15 * meses / 12, 2)
        else:
            raise ValueError("Producto no soportado")


for p in ["ahorros", "cdt", "libranza"]:
    c = CuentaSinPatron(1_000_000, p)
    print(f"{p:10} 12 meses -> interes ${c.calcular_interes(12)}")
print(">> Problema: agregar un producto obliga a MODIFICAR calcular_interes (viola Open/Closed).")

ahorros    12 meses -> interes $20000.0
cdt        12 meses -> interes $80000.0
libranza   12 meses -> interes $150000.0
>> Problema: agregar un producto obliga a MODIFICAR calcular_interes (viola Open/Closed).


### Análisis del problema
- La cuenta conoce **todas** las fórmulas: tiene más de una razón para cambiar.
- Agregar un producto (p. ej. **hipotecario**) obliga a **editar** `calcular_interes`.
- No se puede cambiar la fórmula de una cuenta **en tiempo de ejecución**.

## Código *con patrón* (problema resuelto)
Definimos una interfaz `EstrategiaInteres` y una estrategia concreta por producto. La
`Cuenta` recibe una estrategia y delega el cálculo; se puede **intercambiar** en runtime.

In [7]:
from abc import ABC, abstractmethod


class EstrategiaInteres(ABC):
    @abstractmethod
    def calcular(self, saldo: float, meses: int) -> float: ...


class InteresAhorros(EstrategiaInteres):
    def calcular(self, saldo, meses): return round(saldo * 0.02 * meses / 12, 2)

class InteresCDT(EstrategiaInteres):
    def calcular(self, saldo, meses): return round(saldo * 0.08 * meses / 12, 2)

class InteresLibranza(EstrategiaInteres):
    def calcular(self, saldo, meses): return round(saldo * 0.15 * meses / 12, 2)


class Cuenta:
    def __init__(self, saldo: float, estrategia: EstrategiaInteres):
        self.saldo = saldo
        self.estrategia = estrategia

    def calcular_interes(self, meses: int) -> float:
        return self.estrategia.calcular(self.saldo, meses)


cuenta = Cuenta(1_000_000, InteresAhorros())
print("ahorros  10 meses -> $", cuenta.calcular_interes(10))

# Cambiamos la estrategia en tiempo de ejecucion
cuenta.estrategia = InteresCDT()
print("cdt      12 meses -> $", cuenta.calcular_interes(12))

cuenta.estrategia = InteresLibranza()
print("libranza 12 meses -> $", cuenta.calcular_interes(12))
print(">> Solucion: cada formula es una estrategia; agregar un producto es crear una clase nueva.")

ahorros  10 meses -> $ 16666.67
cdt      12 meses -> $ 80000.0
libranza 12 meses -> $ 150000.0
>> Solucion: cada formula es una estrategia; agregar un producto es crear una clase nueva.


### Verificación
- Hay un **contexto** (`Cuenta`), una **interfaz de estrategia** (`EstrategiaInteres`) y
  **3 estrategias concretas**.
- La estrategia se **cambia en tiempo de ejecución** (`cuenta.estrategia = ...`).
- Un producto nuevo es una clase nueva: **Open/Closed** cumplido.

## UML del patrón Strategy
```plantuml
@startuml
class Cuenta {
    - estrategia : EstrategiaInteres
    + calcular_interes(meses)
}
interface EstrategiaInteres {
    + calcular(saldo, meses)
}
EstrategiaInteres <|.. InteresAhorros
EstrategiaInteres <|.. InteresCDT
EstrategiaInteres <|.. InteresLibranza
Cuenta --> EstrategiaInteres
@enduml
```

## ¿Por qué Strategy y no otro patrón?
- El problema es tener **varios algoritmos intercambiables** (una fórmula por producto) y
  poder elegir/cambiar en runtime. Ese es el propósito de Strategy.
- No es Chain of Responsibility: aquí no se **recorre** una cadena buscando quién atiende;
  se selecciona **una** estrategia y se usa.
- No es Factory (creacional): no nos interesa **crear** productos, sino **variar el
  comportamiento** de cálculo de una cuenta ya existente.